# Physical Concept Directions


## Load model and data


In [1]:
# ruff: noqa: E402
import json
import warnings

from tqdm import TqdmWarning
warnings.filterwarnings("ignore", category=TqdmWarning)

import torch

from jetdirections.config import CHECKPOINT_DIR, DROPOUT, MODEL_DIM, NUM_CLASSES, NUM_HEADS, NUM_LAYERS, PROJ_ROOT, VAL_DATA_DIR
from jetdirections.data.dataset import JetClassDataset
from jetdirections.data.sampling import load_class_batch
from jetdirections.data.vectorize import FEATURE_NAMES
from jetdirections.interp import extract_cls_activations, matched_tertile_contrast
from jetdirections.model.transformer import JetTagTransformer
from jetdirections.training.metrics import CLASS_NAMES

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LAYERS = [1, 2, 3, 4]
N_QCD = 30_000
RESULTS_DIR = PROJ_ROOT / "experiments" / "results" / "physical_concept_steering"
DIRECTIONS_DIR = RESULTS_DIR / "directions"
DIRECTIONS_DIR.mkdir(parents=True, exist_ok=True)

checkpoint = torch.load(CHECKPOINT_DIR / "jetclass_transformer.pt", map_location=DEVICE, weights_only=False)
model = JetTagTransformer(
    input_dim=17, embed_dim=MODEL_DIM, num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS, num_classes=NUM_CLASSES, dropout=DROPOUT,
).to(DEVICE)
state = checkpoint["model_state_dict"] if "model_state_dict" in checkpoint else checkpoint
model.load_state_dict(state)
model.eval()

dataset = JetClassDataset(VAL_DATA_DIR, shuffle_files=False)
qcd = load_class_batch(dataset, "QCD", N_QCD, seed=0)


## Define physical quantities


In [2]:
PT, CHARGE, D0, D0ERR, DETA, DPHI = [FEATURE_NAMES.index(name) for name in (
    "part_pt_log", "part_charge", "part_d0", "part_d0err", "part_deta", "part_dphi"
)]

def jet_pt(x, mask):
    pt = torch.exp(x[:, PT] / 0.7 + 1.7)
    return (pt * ~mask).sum(dim=1)

def jet_mass(x, mask):
    pt = torch.exp(x[:, PT] / 0.7 + 1.7)
    deta, dphi = x[:, DETA], x[:, DPHI]
    energy = pt * torch.cosh(deta)
    pz = pt * torch.sinh(deta)
    px, py = pt * torch.cos(dphi), pt * torch.sin(dphi)
    real = (~mask).unsqueeze(1)
    total = torch.stack([energy, px, py, pz], dim=1) * real
    summed = total.sum(dim=2)
    return torch.sqrt((summed[:, 0] ** 2 - (summed[:, 1:] ** 2).sum(dim=1)).clamp_min(0))

def multiplicity(mask):
    return (~mask).sum(dim=1).float()

def jet_charge(x, mask):
    pt = torch.exp(x[:, PT] / 0.7 + 1.7)
    real_pt = pt * ~mask
    return (x[:, CHARGE] * real_pt).sum(dim=1) / real_pt.sum(dim=1).clamp_min(1e-6)

def displacement(x, mask):
    charged = (x[:, CHARGE] != 0) & ~mask
    d0 = torch.atanh(x[:, D0].clamp(-0.999999, 0.999999)).abs()
    significance = d0 / x[:, D0ERR].clamp_min(1e-6)
    return torch.log1p((significance * charged).sum(dim=1))

x, mask, labels = qcd
quantities = {
    "mass": jet_mass(x, mask),
    "multiplicity": multiplicity(mask),
    "charge": jet_charge(x, mask),
    "displacement": displacement(x, mask),
}
confounders = {
    "mass": [quantities["multiplicity"], jet_pt(x, mask)],
    "multiplicity": [quantities["mass"], jet_pt(x, mask)],
    "charge": [quantities["mass"], quantities["multiplicity"], jet_pt(x, mask)],
    "displacement": [quantities["mass"], quantities["multiplicity"], jet_pt(x, mask)],
}


## Extract matched directions


In [3]:
def contrast_batches(target, confounder_values):
    low_idx, high_idx = matched_tertile_contrast(target, confounder_values, n_bins=6, seed=0)
    return tuple(t[low_idx] for t in qcd), tuple(t[high_idx] for t in qcd)

def extract_direction(target, confounder_values):
    low, high = contrast_batches(target, confounder_values)
    low_acts = extract_cls_activations(model, low, LAYERS)
    high_acts = extract_cls_activations(model, high, LAYERS)
    return {layer: high_acts[layer].mean(dim=0) - low_acts[layer].mean(dim=0) for layer in LAYERS}

directions = {
    name: extract_direction(quantities[name], confounders[name])
    for name in ("mass", "multiplicity", "charge", "displacement")
}


## Remove correlated components from displacement


In [4]:
def residualize(vector, bases):
    orthonormal = []
    for base in bases:
        component = base.clone()
        for previous in orthonormal:
            component -= torch.dot(component, previous) * previous
        component /= component.norm()
        orthonormal.append(component)
    result = vector.clone()
    for component in orthonormal:
        result -= torch.dot(result, component) * component
    return result

for layer in LAYERS:
    directions["displacement"][layer] = residualize(
        directions["displacement"][layer],
        [directions[name][layer] for name in ("mass", "multiplicity", "charge")],
    )


## Save directions


In [5]:
for name, vectors in directions.items():
    torch.save(vectors, DIRECTIONS_DIR / f"{name}_direction.pt")

class_batches = {c: load_class_batch(dataset, c, 5_000, seed=0) for c in CLASS_NAMES}
class_orders = {
    name: sorted(
        CLASS_NAMES,
        key=lambda c: quantities_fn(class_batches[c][0], class_batches[c][1]).mean().item(),
    )
    for name, quantities_fn in {
        "mass": jet_mass,
        "multiplicity": lambda x, mask: multiplicity(mask),
        "charge": jet_charge,
        "displacement": displacement,
    }.items()
}
(DIRECTIONS_DIR / "class_orders.json").write_text(json.dumps(class_orders, indent=2))


532